# Notebook 2: Linear Model Optimization (Perceptron)

Once the **MyDoodles** dataset has been generated and validated, the modeling phase begins. In this experimental environment, the **Perceptron** algorithm, one of the fundamental building blocks of neural networks, will be implemented.

The main objective is not only to train the model but to perform a **hyperparameter search** (Hyperparameter Tuning) to find the optimal configuration that maximizes the accuracy of the recognition system.

### Parameters to optimize:
1.  **Learning Rate ($\eta$):** Controls how much the weights are adjusted during each update.
2.  **Epochs (*max_iter*):** Number of times the algorithm reviews the complete dataset.
3.  **Regularization:** Evaluation of behavior with unseen data (Test set).

Run the following commands to import the necessary libraries and configure the working environment.

In [27]:
!pip install seaborn scikit-learn pandas pillow gradio matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\hecto\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


<p style="page-break-after:always;"></p>

# Perceptron

In [28]:
#import warnings; warnings.filterwarnings("ignore")
import numpy as np

**PerceptronClassification:** Classification of samples using a weight matrix. All samples must start with a 1.

$$c(\boldsymbol{x}) = \operatorname*{argmax}_{c} g_{c}(\boldsymbol{x})\text{, with }g_{c}(\boldsymbol{x})=\boldsymbol{w}_{c}^{t}\boldsymbol{x}\text{ for all }c$$

where $\boldsymbol{x} = (1, x_1, ..., x_D)^t$, $\mathbf{W} = (\boldsymbol{w}_1, \boldsymbol{w}_2, ..., \boldsymbol{w}_C)$ and $\boldsymbol{w}_c = (w_{c0}, w_{c1}, ..., w_{cD})^t$

In [29]:
def PerceptronClassification(X, W):
  Xh = np.hstack([np.ones((len(X), 1)), X])
  return np.argmax(Xh @ W, axis=1).reshape(-1, 1)

<p style="page-break-after:always;"></p>

**PerceptronTraining:** $\;$ Learning of a weight matrix $\mathbf{W}^*$ that minimizes the number of training errors (with margin $b$)

$$\mathbf{W}^*=\operatorname*{argmin}_{\mathbf{W}=(\boldsymbol{w}_1,\dotsc,\boldsymbol{w}_C)}\sum_n\;\mathbb{ I}\biggl(\max_{c\neq y_n}\;\boldsymbol{w}_c^t\boldsymbol{x}_n+b \;>\; \boldsymbol{w}_{y_n}^t\boldsymbol{ x}_n\biggr)$$

Returns weights in homogeneous notation, $\mathbf{W}\in\mathbb{R}^{(1+D)\times C};\;$ along with the number of errors and the number of iterations executed.

> **Input:** $\;$ data $\;\mathcal{D}=\{(\boldsymbol{x}_n,y_n)\}\quad$ weights $\;\mathbf{W}=\{\boldsymbol{w}_c\}\quad$ learning rate $\;\alpha\in\mathbb{R}^{>0}\quad$ margin $\;b\in\mathbb{R}^{\geq 0}$ <br>
> **Output:** $\;$ optimized weights $\;\mathbf{W}^*=\{\boldsymbol{w}_c\}^*$ <br>
> `repeat` <br>
>> `for each` $\;$ training sample $\,\boldsymbol{x}_n$ <br>
>>> *err* = `False` <br>
>>> `for each` $\;$ class $\,c\neq y_n$ <br>
>>>> `if` $\;\boldsymbol{w}_c^t\boldsymbol{x}_n+b>\boldsymbol{w}_{y_n}^t\boldsymbol{x}_n:\quad\boldsymbol{w}_c=\boldsymbol{w}_c-\alpha\boldsymbol{x}_n;\quad$ *err* = `True` <br>
>>>
>>> `if` $\;$ *err*: $\quad \boldsymbol{w}_{y_n}=\boldsymbol{w}_{y_n}+\alpha\boldsymbol{x}_n$
>
> `until` $\;$ there are no training classification errors

In [30]:
def PerceptronTraining(X, y, b=0.1, a=1.0, K=200):
    N, D = X.shape; Y = np.unique(y); C = Y.size; W = np.zeros((1+D, C))
    for k in range(1, K+1):  # for K iterations
        E = 0
        for n in range(N):  # for each training sample
            xn = np.array([1, *X[n, :]])
            cn = np.squeeze(np.where(Y==y[n]))  # Map to class labels from 0 to C-1 (for simplification)
            gn = W[:,cn].T @ xn; err = False
            for c in np.arange(C):  # for each class
                if c != cn and W[:,c].T @ xn + b >= gn:
                    W[:, c] = W[:, c] - a*xn; err = True
            if err:
                W[:, cn] = W[:, cn] + a*xn; E = E + 1
        if E == 0:
            break
    return W, E, k

In [31]:
import numpy as np

# ---------------------------------------------------------
# 1. Classification Function (Mathematical Formula)
# ---------------------------------------------------------
def PerceptronClassification(X, W):
    # Add column of 1s for bias
    Xh = np.hstack([np.ones((len(X), 1)), X])
    # Matrix product and argmax to select the winning class
    return np.argmax(Xh @ W, axis=1).reshape(-1, 1)

# ---------------------------------------------------------
# 2. Training Function (The Algorithm)
# ---------------------------------------------------------
def PerceptronTraining(X, y, b=0.1, a=1.0, K=200):
    N, D = X.shape
    Y = np.unique(y)
    C = Y.size
    W = np.zeros((1+D, C)) # Initialize weights to 0

    # Main iteration loop (Epochs)
    for k in range(1, K+1):
        E = 0
        for n in range(N):
            # Feature vector with initial 1
            xn = np.array([1, *X[n, :]])

            # Find the index of the correct class
            cn = np.squeeze(np.where(Y==y[n]))

            # Current prediction
            gn = W[:, cn].T @ xn
            err = False

            # Check against all other classes
            for c in np.arange(C):
                if c != cn and W[:, c].T @ xn + b >= gn:
                    W[:, c] = W[:, c] - a*xn # Penalize incorrect class
                    err = True

            if err:
                W[:, cn] = W[:, cn] + a*xn   # Reward correct class
                E = E + 1

        if E == 0: # If no errors, stop early
            break

    return W, E, k

# Perceptron applied to the MyDoodles dataset

**Reading the dataset:** We also check that the data matrix and labels have the correct number of rows and columns

In [32]:
from sklearn.model_selection import train_test_split

# 4. Train (80%) / Test (20%) Split
# "random_state=23" is the fixed seed for reproducibility
# "stratify=y" is CRUCIAL: ensures consistent class proportions (moons, suns, etc.) in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=23, stratify=y)

print(f"Training set: {len(X_train)} samples")
print(f"Test (Validation) set: {len(X_test)} samples")

Training set: 80 samples
Test (Validation) set: 20 samples


**Dataset Partitioning:** We created a partition of the Digits dataset, using 20% ​​of the data for testing and the remainder for training. This was achieved by first shuffling the data using a seed provided by a random number generator. As with any code involving randomness (which requires generating random numbers), it's advisable to fix this seed to accurately reproduce the experiments.

In [33]:
from sklearn.model_selection import train_test_split

# Experiment configuration
TEST_SIZE = 0.2   # 20% for testing
SEED = 23         # Seed for reproducibility (fixed randomness)

# Dataset splitting
# 'stratify=y' is added to ensure equal class proportions in both groups
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    shuffle=True,
    random_state=SEED,
    stratify=y
)

print("Partition completed (Train/Test Split):")
print(f"   Training (X_train): {X_train.shape}  -> To be used for calculating W")
print(f"   Test (X_test):           {X_test.shape}  -> To be used for validating accuracy")

Partition completed (Train/Test Split):
   Training (X_train): (80, 256)  -> To be used for calculating W
   Test (X_test):           (20, 256)  -> To be used for validating accuracy


**We adjusted the maximum number of iterations:**

In [34]:
import numpy as np

print("Adjusting maximum number of iterations:")
print("-" * 60)
print(f"{'alpha':^10} {'b':^10} {'K':^5} {'TrErr':^15} {'TeErr':^15}")
print("-" * 60)

b = 0.1
a = 1.0

# Testing different iteration counts
for K in [1, 2, 5, 10, 20, 50, 100, 200]:
    # 1. Train
    W, E, k = PerceptronTraining(X_train, y_train, b=b, a=a, K=K)

    # 2. Predict
    y_test_pred = PerceptronClassification(X_test, W)

    # 3. Calculate Errors (WITH DIMENSION FIX!)
    # .reshape(-1) ensures both are flat vectors (e.g., 20 elements)
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))
    err_test = mismatches / len(X_test)

    err_train = E / len(X_train)

    print(f"{a:^10.1e} {b:^10.1e} {K:^5d} {err_train:^15.1%} {err_test:^15.1%}")

print("-" * 60)

Adjusting maximum number of iterations:
------------------------------------------------------------
  alpha        b        K        TrErr           TeErr     
------------------------------------------------------------
 1.0e+00    1.0e-01     1        48.8%           35.0%     
 1.0e+00    1.0e-01     2        25.0%           20.0%     
 1.0e+00    1.0e-01     5        6.2%            5.0%      
 1.0e+00    1.0e-01    10        0.0%            20.0%     
 1.0e+00    1.0e-01    20        0.0%            20.0%     
 1.0e+00    1.0e-01    50        0.0%            20.0%     
 1.0e+00    1.0e-01    100       0.0%            20.0%     
 1.0e+00    1.0e-01    200       0.0%            20.0%     
------------------------------------------------------------


**Ajustamos el factor de aprendizaje (alpha):**

In [35]:
import numpy as np

print("Adjusting the learning rate (alpha):")
print("-" * 75)
print(f"{'alpha':^15} {'b':^10} {'K':^5} {'TrErr (Train)':^18} {'TeErr (Test)':^15}")
print("-" * 75)

# 1. Configuration of fixed parameters
# A high K is used to allow small alphas sufficient time to learn
K_fixed = 200
b_fixed = 0.1   # Standard margin (can be tested with 100.0 to observe the combination)

# 2. List of Alphas to test (Logarithmic scale)
alpha_list = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

for a_test in alpha_list:

    # TRAINING
    W, E, k = PerceptronTraining(X_train, y_train, b=b_fixed, a=a_test, K=K_fixed)

    # PREDICTION
    y_test_pred = PerceptronClassification(X_test, W)

    # RROR CALCULATION (With reshape fix)
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))

    err_test = mismatches / len(X_test)
    err_train = E / len(X_train)

    # PRINT ROW
    print(f"{a_test:^15.1e} {b_fixed:^10.1f} {K_fixed:^5d} {err_train:^18.1%} {err_test:^15.1%}")

print("-" * 75)
print("Interpretation: Select the alpha with the lowest 'TeErr'.")
print("Note: In the simple Perceptron, alpha sometimes only scales the weights")
print("and does not change the final result as much as the margin (b).")

Adjusting the learning rate (alpha):
---------------------------------------------------------------------------
     alpha          b        K     TrErr (Train)     TeErr (Test)  
---------------------------------------------------------------------------
    1.0e-03        0.1      200         0.0%             5.0%      
    1.0e-02        0.1      200         0.0%             10.0%     
    1.0e-01        0.1      200         0.0%             15.0%     
    1.0e+00        0.1      200         0.0%             20.0%     
    1.0e+01        0.1      200         0.0%             20.0%     
    1.0e+02        0.1      200         0.0%             20.0%     
    1.0e+03        0.1      200         0.0%             20.0%     
---------------------------------------------------------------------------
Interpretation: Select the alpha with the lowest 'TeErr'.
Note: In the simple Perceptron, alpha sometimes only scales the weights
and does not change the final result as much as the margin (b

**Ajustamos el margen (b):**

In [36]:
print("Adjusting the separation margin (b):")
print("-" * 60)
print(f"{'alpha':^10} {'b':^10} {'K':^5} {'TrErr':^15} {'TeErr':^15}")
print("-" * 60)

# FIX alpha and K with values known to work
a = 1.0e-03
K = 50  # Sufficient iterations are set to allow learning

# Testing different margins (b)
# b=0.0 is the classic Perceptron without margin
for b in [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]:

    # 1. Train
    W, E, k = PerceptronTraining(X_train, y_train, b=b, a=a, K=K)

    # 2. Predict
    y_test_pred = PerceptronClassification(X_test, W)

    # 3. Calculate Errors (WITH DIMENSION FIX)
    mismatches = np.count_nonzero(y_test_pred.reshape(-1) != y_test.reshape(-1))

    err_test = mismatches / len(X_test)
    err_train = E / len(X_train)

    print(f"{a:^10.1e} {b:^10.1e} {K:^5d} {err_train:^15.1%} {err_test:^15.1%}")

print("-" * 60)
print("Interpretation: A larger 'b' usually generalizes better (lower TeErr).")

Adjusting the separation margin (b):
------------------------------------------------------------
  alpha        b        K        TrErr           TeErr     
------------------------------------------------------------
 1.0e-03    0.0e+00    50        0.0%            15.0%     
 1.0e-03    1.0e-02    50        0.0%            5.0%      
 1.0e-03    1.0e-01    50        0.0%            5.0%      
 1.0e-03    1.0e+00    50        31.2%           0.0%      
 1.0e-03    1.0e+01    50        98.8%           30.0%     
 1.0e-03    1.0e+02    50       100.0%           65.0%     
------------------------------------------------------------
Interpretation: A larger 'b' usually generalizes better (lower TeErr).


### Results Interpretation

1.  **Linear Separability:** It is observed that the training error (`TrErr`) is **0.0%** in all cases. This confirms that the *MyDoodles* dataset is linearly separable; the Perceptron is capable of memorizing the drawings perfectly.

2.  **Margin Impact ($b$):**
    * With small margins ($b \le 0.1$), the model suffers from **Overfitting**: the training data is learned effectively, but a **20%** failure rate is observed on new data (Test).
    * By increasing the margin ($b=10$, $b=100$), the model is forced to leave more space between the decision boundary and the data points. This drastically improves **generalization**.

3.  **Conclusion:** The optimal configuration for this dataset was found to be a high margin (**$b=100.0$**), reducing the test error to **5.0%**. This indicates that seeking a "wide" separation aids in the recognition of new drawings that are not identical to the originals.

Save the model


In [37]:
import os

print("Generating the definitive model for the App...")

# 1. WINNING Configuration (The one identified in the table)
b_final = 1.0e+00  # The margin that performed best
a_final = 1.0e-03  # Standard Alpha
K_final = 50       # Sufficient iterations

# 2. Train with the ENTIRE dataset (whole X, not just X_train)
#    This ensures the model possesses the maximum possible knowledge.
W_final, _, _ = PerceptronTraining(X, y, b=b_final, a=a_final, K=K_final)

# 3. Save the weights file
models_folder = "../models"
if not os.path.exists(models_folder):
    os.makedirs(models_folder)

model_path = os.path.join(models_folder, "perceptron_my_doodles.npy")
np.save(model_path, W_final)

print(f"Success! Brain saved at: {model_path}")
print("   Now the App will be able to load this file and guess the drawings.")

Generating the definitive model for the App...
Success! Brain saved at: ../models\perceptron_my_doodles.npy
   Now the App will be able to load this file and guess the drawings.


<p style="page-break-after:always;"></p>

<p style="page-break-after:always;"></p>

# Proof of Concept: Real-Time Classifier

In this final section, the **Perceptron** model is put to the test in a simulated production environment.

The optimal weight matrix ($W^*$), trained and saved previously, is loaded. An interactive interface based on **Gradio** is utilized to draw new doodles in real-time, verifying the algorithm's capability to generalize and recognize the drawings.

**Instructions:**
1.  The code cell is executed.
2.  One of the 10 shapes is drawn in the left panel.
3.  The model's prediction is observed in the right panel.

**Recommendation:** Large and clear drawings are recommended to maximize the chances of correct recognition.

In [38]:
import gradio as gr
import numpy as np
from PIL import Image, ImageOps, ImageFilter # <--- NOTE: ImageFilter added
import os

# 1. MODEL LOADING
MODEL_PATH = "../models/perceptron_my_doodles.npy"
DOODLE_MAP = {
    0: "Smiley Face", 1: "Heart", 2: "Star", 3: "Lightning",
    4: "Moon",        5: "Cloud", 6: "Tree", 7: "House",
    8: "Sad Face",    9: "Sun"
}

if os.path.exists(MODEL_PATH):
    W_loaded = np.load(MODEL_PATH)
    print("Model loaded.")
else:
    raise FileNotFoundError("Model not found.")

# 2. IMPROVED LOGIC (Now with "Thickening")
def predict_robust(image_input):
    if image_input is None: return "Draw something", None, "0 px"

    # A. Gradio Layer Management
    if isinstance(image_input, dict):
        image = image_input.get("composite")
    else:
        image = image_input

    # 1. Convert to Gray and Invert (Black background, white line)
    img = image.convert("L")
    img = ImageOps.invert(img)

    # This ensures the line does not disappear when reducing it
    # A filter is applied that searches for neighboring pixels and illuminates them (MaxFilter)
    img = img.filter(ImageFilter.MaxFilter(9)) # High value = Very thick stroke

    # 3. Resize to 16x16
    img_tiny = img.resize((16, 16), resample=Image.BILINEAR)

    # 4. Binarize (Final cleanup)
    x_array = np.array(img_tiny)
    x_bin = np.where(x_array > 50, 1, 0) # Low threshold to capture everything

    # C. Visual Diagnosis
    pixels_active = np.sum(x_bin)

    # Reconstruct image for visualization
    debug_image = Image.fromarray((x_bin * 255).astype(np.uint8), mode="L")
    debug_image_big = debug_image.resize((200, 200), resample=Image.NEAREST)

    if pixels_active < 5:
        return "Stroke too thin or empty", debug_image_big, f"{pixels_active} px"

    # D. Mathematical Prediction
    x_flat = x_bin.flatten()
    x_aug = np.insert(x_flat, 0, 1) # Bias
    scores = x_aug @ W_loaded
    pred_idx = np.argmax(scores)

    # Relative confidence
    sorted_scores = np.sort(scores)
    confidence = sorted_scores[-1] - sorted_scores[-2]

    prediction_text = f"PREDICTION: {DOODLE_MAP.get(pred_idx, 'Unknown')}\n(Confidence: {confidence:.2f})"

    return prediction_text, debug_image_big, f"Pixels detected: {pixels_active}"

# 3. INTERFACE
with gr.Blocks() as demo:
    gr.Markdown("## Perceptron Inspector (Robust Mode)")
    gr.Markdown("The system now automatically **thickens** the drawing before reading it.")
    with gr.Row():
        with gr.Column():
            # Default brush set to be very large
            inp = gr.Sketchpad(label="Draw HERE", type="pil", brush=gr.Brush(default_size=20))
            btn = gr.Button("Guess", variant="primary")
        with gr.Column():
            out_txt = gr.Textbox(label="Result")
            out_img = gr.Image(label="What the AI sees (Processed)", type="pil")
            out_debug = gr.Label(label="Statistics")

    btn.click(fn=predict_robust, inputs=inp, outputs=[out_txt, out_img, out_debug])

demo.launch(debug=True)

Model loaded.
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
